# Baseline + Grad-CAM — Chest X-ray Pneumonia Detection

Notebook **độc lập hoàn toàn**: không clone repo, không import gì ngoài thư viện
Kaggle đã có sẵn. Bấm **Run All** là chạy từ đầu đến cuối.

### Notebook này làm gì

Train baseline ResNet18 dưới **hai protocol split song song**, rồi so sánh:

| Protocol | Cắt validation | Dùng để |
|---|---|---|
| **A** `paper_compatible` | stratified mức **ảnh** | So với số published và các notebook tham khảo |
| **B** `patient_grouped` | stratified mức **bệnh nhân** | Số trung thực, dùng để claim generalization |

Cả hai **giữ nguyên test split gốc** làm holdout, nên chỉ ranh giới train/val đổi.
Khoảng cách A − B đo đúng mức mà split theo ảnh thổi phồng validation metric.

Sau đó dùng **Grad-CAM** kiểm tra model nhìn vào đâu — model đúng vì lý do sai
thì vẫn phải loại.

### Trước khi chạy
1. **+ Add Data** → thêm dataset `chest-xray-pneumonia` (paultimothymooney)
2. **Settings → Accelerator** → **GPU** (T4 hoặc P100)
3. **Settings → Internet** → **On** (để tải ImageNet weights cho ResNet18)

## 0. Cấu hình — mọi thứ cần chỉnh đều ở đây

In [ ]:
SEED          = 42
IMG_SIZE      = 224
BATCH_SIZE    = 32
EPOCHS        = 8         # ~4-6 phút/protocol trên T4
LR            = 1e-4
WEIGHT_DECAY  = 1e-5
VAL_FRACTION  = 0.15      # cắt ra từ pool train+val gốc
PATIENCE      = 4         # early stopping theo val F1
CLASS_WEIGHTS = True      # train split lệch ~2.9:1 PNEUMONIA:NORMAL
BORDER_FRAC   = 0.15      # dùng ở phần kiểm tra Grad-CAM

CLASSES     = ("NORMAL", "PNEUMONIA")   # NORMAL=0, PNEUMONIA=1 (positive)
CLASS_TO_ID = {name: index for index, name in enumerate(CLASSES)}

## 1. Môi trường

In [ ]:
import hashlib, os, random, re, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

warnings.filterwarnings("ignore", category=UserWarning)

# Fail loudly rather than silently crawling on CPU for an hour.
assert torch.cuda.is_available(), (
    "Chưa bật GPU. Settings -> Accelerator -> GPU T4/P100, rồi Run All lại."
)
DEVICE = torch.device("cuda")


def set_seed(seed=SEED):
    """Seed every RNG the pipeline touches."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
print("torch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

## 2. Tìm dataset

**Không hardcode đường dẫn.** Kaggle mount dataset này theo **hai kiểu khác nhau**
tuỳ cách bạn thêm nó, cả hai đều gặp trong thực tế:

```
/kaggle/input/chest-xray-pneumonia/chest_xray/...                              (kiểu 1)
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/...   (kiểu 2)
```

Bên trong còn **hai cái bẫy**:

- `chest_xray/chest_xray/` — cây lồng giữ **bản sao thứ hai của toàn bộ ảnh**.
  Quét đệ quy ngây thơ sẽ đếm 11.712 thay vì 5.856.
- `__MACOSX/chest_xray/train/NORMAL/._IM-0748-0001.jpeg` — file resource-fork của
  macOS. Đuôi `.jpeg` nhưng **không phải ảnh**, và cây `__MACOSX/` cũng có đủ
  `train/NORMAL` + `train/PNEUMONIA` nên qua mặt được kiểm tra thư mục mốc.

Hàm dưới xử lý cả bốn trường hợp: dò theo thư mục mốc, loại `__MACOSX`, chọn cây
**nông nhất** (nên bỏ qua cây lồng), và lọc file `._*`.

In [ ]:
def list_images(directory):
    """List real .jpeg files, skipping macOS AppleDouble sidecars (._*)."""
    return sorted(p for p in Path(directory).glob("*.jpeg")
                  if not p.name.startswith("._"))


def find_data_root(search_paths, verbose=True):
    """Locate the directory directly containing train/NORMAL and train/PNEUMONIA.

    Handles both Kaggle mount conventions, skips the __MACOSX sidecar tree, and
    prefers the shallowest match so a nested duplicate tree is never selected.
    """
    if isinstance(search_paths, (str, Path)):
        search_paths = [search_paths]

    candidates = []
    for base in map(Path, search_paths):
        if not base.exists():
            continue
        for train_dir in base.rglob("train"):
            if "__MACOSX" in train_dir.parts:
                continue
            if (train_dir / "NORMAL").is_dir() and (train_dir / "PNEUMONIA").is_dir():
                candidates.append(train_dir.parent)

    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy dataset dưới {[str(p) for p in search_paths]}.\n"
            "Bấm '+ Add Data' ở sidebar và thêm 'Chest X-Ray Images (Pneumonia)' "
            "của paultimothymooney."
        )

    candidates = sorted(set(candidates), key=lambda path: len(path.parts))
    chosen = candidates[0]

    if verbose:
        print("Các cây tìm thấy:")
        for candidate in candidates:
            count = len(list_images(candidate / "train" / "NORMAL"))
            mark = "  <- DÙNG CÁI NÀY" if candidate == chosen else "  (bản trùng, bỏ qua)"
            print(f"  {candidate}  [{count} ảnh train/NORMAL]{mark}")
    return chosen


DATA_ROOT = find_data_root(["/kaggle/input", "../chest_xray", "data/raw"])
print("\nDATA_ROOT =", DATA_ROOT)

# Show that the sidecar tree is really there and really excluded.
sidecars = [p for p in Path(DATA_ROOT).rglob("._*.jpeg")]
if sidecars:
    print(f"Bỏ qua {len(sidecars):,} file resource-fork macOS (._*.jpeg).")

## 3. Manifest và định danh bệnh nhân

Tên file là **thứ duy nhất** cho biết ảnh nào thuộc bệnh nhân nào:

- `person<N>_bacteria_<M>.jpeg` / `person<N>_virus_<M>.jpeg`
- `IM-<N>-<M>.jpeg` / `NORMAL<K>-IM-<N>-<M>.jpeg`

**Điểm mấu chốt:** counter `person` **không toàn cục**. `bacteria` và `virus` mỗi
loại chạy một dãy riêng từ 1, nên `person1_bacteria` và `person1_virus` là **hai
người khác nhau**. Phần 4 sẽ chứng minh bằng số.

In [ ]:
PNEUMONIA_RE = re.compile(r"^person(\d+)_(bacteria|virus)_", re.IGNORECASE)
NORMAL_RE    = re.compile(r"^(?:(NORMAL\d+)-)?IM-(\d+)-", re.IGNORECASE)


def parse_group_id(filename):
    """Derive a patient key from a filename. Raises on unknown patterns."""
    match = PNEUMONIA_RE.match(filename)
    if match:
        return f"pneumonia:{match.group(2).lower()}:{int(match.group(1))}"

    match = NORMAL_RE.match(filename)
    if match:
        return f"normal:{(match.group(1) or 'IM').lower()}:{int(match.group(2))}"

    raise ValueError(f"Tên file lạ, không suy ra được bệnh nhân: {filename}")


def build_manifest(root):
    """Scan the dataset into one row per image."""
    rows = []
    for split in ("train", "val", "test"):
        for class_id, class_name in enumerate(CLASSES):
            directory = Path(root) / split / class_name
            if not directory.is_dir():
                continue
            for path in list_images(directory):     # skips ._* sidecars
                rows.append({
                    "path": str(path),
                    "filename": path.name,
                    "split_original": split,
                    "class_name": class_name,
                    "class_id": class_id,
                    "group_id": parse_group_id(path.name),
                })
    if not rows:
        raise FileNotFoundError(f"Không có ảnh .jpeg nào dưới {root}")
    return pd.DataFrame(rows)


manifest = build_manifest(DATA_ROOT)
print(f"{len(manifest):,} ảnh | {manifest['group_id'].nunique():,} bệnh nhân")
manifest.head(3)

## 4. Data audit

In ra mọi con số để reviewer tự kiểm chứng thay vì phải tin một bảng có sẵn.
Phần cuối là quan trọng nhất: nó quyết định **test split gốc có dùng làm holdout
sạch được hay không**.

In [ ]:
print("=" * 66)
print("4.1  Số lượng theo split và class")
print("=" * 66)
print(f"{'split':<8}{'NORMAL':>9}{'PNEUMONIA':>12}{'total':>9}{'P/N':>7}")
for split in ("train", "val", "test"):
    subset = manifest[manifest["split_original"] == split]
    counts = subset["class_name"].value_counts()
    normal, pneumonia = int(counts.get("NORMAL", 0)), int(counts.get("PNEUMONIA", 0))
    ratio = pneumonia / normal if normal else float("nan")
    print(f"{split:<8}{normal:>9,}{pneumonia:>12,}{normal + pneumonia:>9,}{ratio:>7.2f}")
print(f"{'TOTAL':<8}{'':>9}{'':>12}{len(manifest):>9,}")
print("\nval gốc chỉ 16 ảnh -> không thể chọn checkpoint trên đó. Phần 5 cắt val thật.")

In [ ]:
print("=" * 66)
print("4.2  Ảnh trùng nội dung (SHA-256)")
print("=" * 66)
by_hash = defaultdict(list)
for path, split in zip(manifest["path"], manifest["split_original"]):
    by_hash[hashlib.sha256(Path(path).read_bytes()).hexdigest()].append(split)

dup_groups = [splits for splits in by_hash.values() if len(splits) > 1]
cross_split = [splits for splits in dup_groups if len(set(splits)) > 1]
print(f"tổng file            : {len(manifest):,}")
print(f"hash nội dung duy nhất: {len(by_hash):,}")
print(f"nhóm trùng            : {len(dup_groups)}")
print(f"  trong đó xuyên split: {len(cross_split)}")
print("=> 0 nhóm xuyên split nghĩa là không có ảnh y hệt nằm cả train lẫn test."
      if not cross_split else "=> CẢNH BÁO: ảnh y hệt nằm ở nhiều split, evaluation bị nhiễm.")

In [ ]:
print("=" * 66)
print("4.3  Mode ảnh và độ phân giải")
print("=" * 66)
modes, sizes, unreadable = Counter(), Counter(), 0
for path in manifest["path"]:
    try:
        with Image.open(path) as image:
            modes[image.mode] += 1
            sizes[image.size] += 1
    except OSError:
        unreadable += 1

print(f"file không mở được : {unreadable}")
for mode, count in modes.most_common():
    print(f"  mode {mode:<4}       : {count:,}")
print(f"kích thước khác nhau: {len(sizes):,}")
print("\n-> có ảnh lưu RGB lẫn grayscale, nên pipeline phải ép về 'L' trước,")
print("   không được giả định mọi file đều single-channel.")

In [ ]:
print("=" * 66)
print("4.4  Bệnh nhân — key ngây thơ vs key đúng")
print("=" * 66)
naive_re = re.compile(r"^(person\d+)_", re.IGNORECASE)
naive, corrected = defaultdict(set), defaultdict(set)
for filename, split in zip(manifest["filename"], manifest["split_original"]):
    match = naive_re.match(filename)
    naive[match.group(1).lower() if match else filename].add(split)
    corrected[parse_group_id(filename)].add(split)

naive_span     = sum(1 for splits in naive.values() if len(splits) > 1)
corrected_span = sum(1 for splits in corrected.values() if len(splits) > 1)
print(f"key ngây thơ  person<N>           : {len(naive):,} nhóm, {naive_span} vắt qua >1 split")
print(f"key đúng      (subtype, person<N>): {len(corrected):,} nhóm, {corrected_span} vắt qua >1 split")

subtype_ids = defaultdict(set)
for filename in manifest["filename"]:
    match = PNEUMONIA_RE.match(filename)
    if match:
        subtype_ids[match.group(2).lower()].add(int(match.group(1)))

print("\nbằng chứng counter chạy riêng theo subtype:")
for subtype, ids in sorted(subtype_ids.items()):
    print(f"  {subtype:<9}: {len(ids):,} id, dải 1..{max(ids)}, mật độ {len(ids)/max(ids):.3f}")
shared = subtype_ids["bacteria"] & subtype_ids["virus"]
print(f"  số person dùng bởi CẢ HAI subtype: {len(shared):,}")
print("  (nếu counter dùng chung thì con số này phải ~0 — thực tế thì không)")

per_group = Counter(parse_group_id(f) for f in manifest["filename"])
multi = sum(1 for n in per_group.values() if n > 1)
print(f"\nnhóm có >1 ảnh: {multi:,}/{len(per_group):,} (nhiều nhất {max(per_group.values())} ảnh/nhóm)")
print("-> cắt val theo ảnh CHẮC CHẮN sẽ xẻ đôi bệnh nhân. Đó là lý do có protocol B.")

print()
if corrected_span == 0:
    print("KẾT LUẬN: train/val/test gốc là patient-disjoint.")
    print("          Test split gốc dùng làm holdout sạch được.")
    print(f"          {naive_span} 'overlap' theo key ngây thơ là artifact, không phải leakage.")
else:
    print(f"KẾT LUẬN: {corrected_span} bệnh nhân vắt qua split -> phải dựng holdout mới.")

## 5. Hai protocol split

Cả hai giữ nguyên test gốc; chỉ ranh giới train/val đổi.
`leaked_patient_groups` của protocol A **được kỳ vọng khác 0** — đó là phát hiện
cần báo cáo, không phải bug.

In [ ]:
def make_splits(manifest, protocol, val_fraction=VAL_FRACTION, seed=SEED):
    """Assign each image to train/val/test. The original test split is kept."""
    manifest = manifest.copy()
    pool = manifest[manifest["split_original"].isin(["train", "val"])]
    test = manifest[manifest["split_original"] == "test"]

    if protocol == "a_paper_compatible":
        train, val = train_test_split(
            pool, test_size=val_fraction,
            stratify=pool["class_id"], random_state=seed,
        )
    elif protocol == "b_patient_grouped":
        splitter = StratifiedGroupKFold(
            n_splits=max(2, round(1 / val_fraction)), shuffle=True, random_state=seed
        )
        train_idx, val_idx = next(
            splitter.split(pool, pool["class_id"], groups=pool["group_id"])
        )
        train, val = pool.iloc[train_idx], pool.iloc[val_idx]
    else:
        raise ValueError(f"Protocol lạ: {protocol!r}")

    manifest["split"] = pd.NA
    manifest.loc[train.index, "split"] = "train"
    manifest.loc[val.index,   "split"] = "val"
    manifest.loc[test.index,  "split"] = "test"
    return manifest


def count_leaked_groups(split_manifest):
    """Number of patients appearing in more than one split."""
    return int((split_manifest.groupby("group_id")["split"].nunique() > 1).sum())


def split_summary(split_manifest):
    """Per-split class counts, patient counts and class ratio."""
    rows = []
    for split in ("train", "val", "test"):
        subset = split_manifest[split_manifest["split"] == split]
        counts = subset["class_name"].value_counts()
        normal, pneumonia = int(counts.get("NORMAL", 0)), int(counts.get("PNEUMONIA", 0))
        rows.append({
            "split": split, "NORMAL": normal, "PNEUMONIA": pneumonia,
            "total": normal + pneumonia,
            "patients": subset["group_id"].nunique(),
            "P/N": round(pneumonia / max(normal, 1), 2),
        })
    return pd.DataFrame(rows).set_index("split")


SPLITS = {}
overview = []
for protocol in ("a_paper_compatible", "b_patient_grouped"):
    SPLITS[protocol] = make_splits(manifest, protocol)
    leaked = count_leaked_groups(SPLITS[protocol])
    print(f"\n=== {protocol} ===")
    print(split_summary(SPLITS[protocol]).to_string())
    print(f"bệnh nhân vắt qua >1 split: {leaked}")
    overview.append({"protocol": protocol, "leaked_patient_groups": leaked})

print()
pd.DataFrame(overview).set_index("protocol")

## 6. Data pipeline

Ép mọi ảnh về grayscale rồi nhân thành 3 kênh cho backbone ImageNet — kể cả
những file vốn đã lưu ở RGB (phần 4.3). Augmentation **chỉ áp cho train**;
val/test phải deterministic, nếu không metric thành nhiễu.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class XRayDataset(Dataset):
    """Dataset driven by an explicit (path, label) list from the manifest."""

    def __init__(self, rows, transform):
        self.rows, self.transform = rows, transform
        self.targets = [label for _, label in rows]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        path, label = self.rows[index]
        return self.transform(Image.open(path).convert("L")), label


def make_loaders(split_manifest, batch_size=BATCH_SIZE):
    """Build train/val/test DataLoaders from a split manifest."""
    loaders = {}
    for split, transform in (("train", train_tf), ("val", eval_tf), ("test", eval_tf)):
        subset = split_manifest[split_manifest["split"] == split]
        rows = list(zip(subset["path"], subset["class_id"]))
        loaders[split] = DataLoader(
            XRayDataset(rows, transform),
            batch_size=batch_size, shuffle=(split == "train"),
            num_workers=2, pin_memory=True,
        )
    return loaders


def class_weights_from(split_manifest):
    """Inverse-frequency weights, scaled so the loss keeps its usual magnitude."""
    counts = Counter(split_manifest[split_manifest["split"] == "train"]["class_id"])
    total = sum(counts.values())
    return torch.tensor(
        [total / (len(CLASSES) * counts[i]) for i in range(len(CLASSES))],
        dtype=torch.float, device=DEVICE,
    )

print("pipeline sẵn sàng.")

## 7. Model, train, evaluate

Hai điểm cần chú ý, vì đây là chỗ hay sai nhất:

1. **Load lại best weights trước khi chấm test.** Nếu không, số test là của epoch
   cuối — với early stopping thì lệch tới `PATIENCE` epoch so với model được chọn.
2. **`best_f1` khởi tạo `-1.0`**, không phải `0.0`. Nếu khởi tạo `0.0` mà F1 luôn
   bằng 0 thì `0.0 > 0.0` sai, không epoch nào lưu checkpoint.

In [ ]:
def build_resnet18():
    """ImageNet-pretrained ResNet18 with a fresh 2-class head."""
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    return model.to(DEVICE)


@torch.no_grad()
def evaluate(model, loader):
    """Accuracy / precision / recall / F1 / AUC / confusion matrix."""
    model.eval()
    labels_all, preds_all, probs_all = [], [], []
    for images, labels in loader:
        logits = model(images.to(DEVICE, non_blocking=True))
        probabilities = torch.softmax(logits.float(), dim=1)[:, 1]
        probs_all += probabilities.cpu().tolist()
        preds_all += logits.argmax(1).cpu().tolist()
        labels_all += labels.tolist()

    return {
        "accuracy":  float(np.mean(np.array(labels_all) == np.array(preds_all))),
        "precision": precision_score(labels_all, preds_all, zero_division=0),
        "recall":    recall_score(labels_all, preds_all, zero_division=0),
        "f1":        f1_score(labels_all, preds_all, zero_division=0),
        "auc":       roc_auc_score(labels_all, probs_all),
        "confusion_matrix": confusion_matrix(labels_all, preds_all).tolist(),
    }


def run_experiment(protocol, epochs=EPOCHS):
    """Train under one split protocol and score the best checkpoint on test."""
    print(f"\n{'=' * 66}\n{protocol}\n{'=' * 66}")
    set_seed()

    split_manifest = SPLITS[protocol]
    loaders = make_loaders(split_manifest)
    model = build_resnet18()

    weights = class_weights_from(split_manifest) if CLASS_WEIGHTS else None
    if weights is not None:
        print("class weights:", [round(w, 3) for w in weights.tolist()])
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.3, patience=2
    )
    scaler = torch.amp.GradScaler("cuda")

    best_f1, best_epoch, best_state, stale = -1.0, 0, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for images, labels in loaders["train"]:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * images.size(0)

        train_loss = running_loss / len(loaders["train"].dataset)
        val_metrics = evaluate(model, loaders["val"])
        scheduler.step(val_metrics["f1"])

        marker = ""
        if val_metrics["f1"] > best_f1:
            best_f1, best_epoch, stale = val_metrics["f1"], epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            marker = "  <- best"
        else:
            stale += 1

        print(f"epoch {epoch:>2}/{epochs}  loss {train_loss:.4f}  "
              f"val_f1 {val_metrics['f1']:.4f}  val_auc {val_metrics['auc']:.4f}{marker}")

        if stale >= PATIENCE:
            print(f"early stopping ở epoch {epoch}")
            break

    # Restore the selected model BEFORE touching test -- otherwise the reported
    # numbers belong to whatever epoch the loop happened to stop on.
    model.load_state_dict(best_state)
    print(f"khôi phục best checkpoint (epoch {best_epoch}, val_f1 {best_f1:.4f})")

    torch.save(best_state, f"/kaggle/working/resnet18_{protocol}.pth")
    split_manifest.to_csv(f"/kaggle/working/manifest_{protocol}.csv", index=False)

    return {
        "model": model,
        "loaders": loaders,
        "best_epoch": best_epoch,
        "val": evaluate(model, loaders["val"]),
        "test": evaluate(model, loaders["test"]),
    }

print("sẵn sàng train.")

## 8. Chạy cả hai protocol

In [ ]:
RESULTS = {protocol: run_experiment(protocol)
           for protocol in ("b_patient_grouped", "a_paper_compatible")}

## 9. So sánh A và B

Đọc bảng này cho đúng, vì rất dễ diễn giải sai:

- **Bằng chứng leakage là con số 239 vs 0** ở phần 5, đo trực tiếp bằng
  `count_leaked_groups`. Đó là sự thật về cấu trúc dữ liệu, không phụ thuộc
  model, seed hay số epoch.
- **Val F1 của A và B không so trực tiếp được** — hai protocol có tập validation
  **khác nhau** (785 vs 748 ảnh, thành phần bệnh nhân khác). Chênh lệch val chỉ
  mang tính mô tả.
- **Chỉ cột `test` mới so được apples-to-apples**, vì test split giống hệt nhau ở
  cả hai. Khác biệt duy nhất là validation nào đã chọn ra checkpoint.
- Đây là **một seed**. Muốn kết luận chắc thì cần 3–5 seed và báo khoảng dao động.

In [ ]:
table = pd.DataFrame([
    {"protocol": protocol, "split": split,
     **{k: round(v, 4) for k, v in result[split].items() if k != "confusion_matrix"}}
    for protocol, result in RESULTS.items() for split in ("val", "test")
]).set_index(["protocol", "split"])
display(table)

test_gap = (RESULTS["a_paper_compatible"]["test"]["f1"]
            - RESULTS["b_patient_grouped"]["test"]["f1"])
print(f"Test F1  A - B = {test_gap:+.4f}   (cùng một test split, so được)")
if test_gap < 0:
    print("  -> Protocol B chọn ra checkpoint tổng quát hoá tốt hơn trên test.")
else:
    print("  -> Lần chạy này A không kém hơn trên test. Với 1 seed, đừng kết luận vội;")
    print("     bằng chứng leakage nằm ở con số 239 bệnh nhân, không ở delta F1 này.")

print("\nVal F1 (KHÔNG so trực tiếp được — hai tập val khác nhau):")
for protocol in ("b_patient_grouped", "a_paper_compatible"):
    n_val = int((SPLITS[protocol]["split"] == "val").sum())
    print(f"  {protocol:<20} {RESULTS[protocol]['val']['f1']:.4f}  trên {n_val:,} ảnh")

for protocol, result in RESULTS.items():
    (tn, fp), (fn, tp) = result["test"]["confusion_matrix"]
    total = tn + fp + fn + tp
    print(f"\n{protocol} — test confusion matrix (n={total}):")
    print(f"  TN {tn:>4}   FP {fp:>4}")
    print(f"  FN {fn:>4}   TP {tp:>4}")
    print(f"  bỏ sót {fn}/{fn + tp} ca viêm phổi | báo động giả {fp}/{tn + fp} ca bình thường")
print("\nFN là ô đáng lo nhất về lâm sàng: bỏ sót viêm phổi nguy hiểm hơn báo nhầm.")

## 10. Explainable AI — Grad-CAM

Một model **đúng vì lý do sai** thì vẫn phải loại. Grad-CAM vẽ vùng ảnh mà model
dựa vào, nên đây là công cụ QA chứ không phải minh hoạ cho đẹp.

Ba lựa chọn có chủ đích ở dưới:

1. **Target layer chỉ đích danh** `model.layer4[-1]`. Cách phổ biến là dò
   `[m for m in model.modules() if isinstance(m, nn.Conv2d)][-1]` — đổi backbone
   là sai **âm thầm**, heatmap vẫn ra nhưng ra bậy.
2. **Lấy mẫu theo từng ô confusion matrix** (TN/FP/FN/TP), thay vì lấy đều theo
   index — dataset sắp theo class nên lấy đều sẽ lệch hẳn về PNEUMONIA.
3. **Tự implement bằng hook**, không `pip install grad-cam` — bớt một điểm hỏng
   giữa lúc demo và không phụ thuộc version thư viện.

In [ ]:
class GradCAM:
    """Grad-CAM via forward/backward hooks on one convolutional layer.

    Each activation channel is weighted by its mean gradient w.r.t. the target
    logit, summed, then rectified -- the regions whose presence pushed the
    score up.
    """

    def __init__(self, model, target_layer):
        self.model, self.activations, self.gradients = model, None, None
        self._handles = [
            target_layer.register_forward_hook(self._save_activations),
            target_layer.register_full_backward_hook(self._save_gradients),
        ]

    def _save_activations(self, module, inputs, output):
        self.activations = output.detach()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, images, class_indices=None):
        """Return (cams as [N,H,W] in 0..1, logits)."""
        self.model.zero_grad(set_to_none=True)
        logits = self.model(images)          # gradients needed, so no no_grad here
        if class_indices is None:
            class_indices = logits.argmax(dim=1)
        logits[torch.arange(len(images)), class_indices].sum().backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cams = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cams = F.interpolate(cams, size=images.shape[-2:],
                             mode="bilinear", align_corners=False)[:, 0]
        cams = cams - cams.amin(dim=(1, 2), keepdim=True)
        cams = cams / (cams.amax(dim=(1, 2), keepdim=True) + 1e-8)
        return cams.cpu().numpy(), logits.detach()

    def close(self):
        for handle in self._handles:
            handle.remove()


def denormalize(tensor):
    """Undo ImageNet normalisation, returning an HxW image in 0..1."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1).mean(0).numpy()


XAI_PROTOCOL = "b_patient_grouped"        # protocol B là kết quả chính
xai_model = RESULTS[XAI_PROTOCOL]["model"].eval()
cam_engine = GradCAM(xai_model, xai_model.layer4[-1])
print("Grad-CAM gắn vào:", XAI_PROTOCOL, "| layer4[-1]")

### 10.1. Chọn một ca cho mỗi ô confusion matrix

Lấy ca model **tự tin nhất** trong mỗi ô — ca tự tin mà vẫn sai là ca đáng mổ xẻ.

In [ ]:
xai_manifest = SPLITS[XAI_PROTOCOL]
test_rows = xai_manifest[xai_manifest["split"] == "test"].reset_index(drop=True)

labels_all, preds_all, probs_all = [], [], []
with torch.no_grad():
    for images, labels in RESULTS[XAI_PROTOCOL]["loaders"]["test"]:
        probabilities = torch.softmax(xai_model(images.to(DEVICE)).float(), dim=1)
        probs_all  += probabilities[:, 1].cpu().tolist()
        preds_all  += probabilities.argmax(1).cpu().tolist()
        labels_all += labels.tolist()

test_rows = test_rows.assign(true=labels_all, pred=preds_all, p_pneumonia=probs_all)

CM_CELLS = {
    "TN — NORMAL đúng":      (0, 0),
    "FP — báo động giả":     (0, 1),
    "FN — BỎ SÓT viêm phổi": (1, 0),
    "TP — PNEUMONIA đúng":   (1, 1),
}

picks = {}
for name, (true_label, pred_label) in CM_CELLS.items():
    subset = test_rows[(test_rows["true"] == true_label) & (test_rows["pred"] == pred_label)]
    if subset.empty:
        print(f"{name:<24}: không có ca nào")
        continue
    confidence = subset["p_pneumonia"] if pred_label == 1 else 1 - subset["p_pneumonia"]
    picks[name] = subset.loc[confidence.idxmax()]
    print(f"{name:<24}: {len(subset):>3} ca  |  chọn {picks[name]['filename']}")

### 10.2. Heatmap

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(2, len(picks), figsize=(4.0 * len(picks), 8.2))
axes = np.atleast_2d(axes)

for column, (name, row) in enumerate(picks.items()):
    tensor = eval_tf(Image.open(row["path"]).convert("L")).unsqueeze(0).to(DEVICE)
    cams, logits = cam_engine(tensor)
    probability = torch.softmax(logits.float(), 1)[0, int(row["pred"])].item()
    grayscale = denormalize(tensor[0])

    axes[0, column].imshow(grayscale, cmap="gray")
    axes[0, column].set_title(f"{name}\nthật: {CLASSES[row['true']]}", fontsize=11)

    axes[1, column].imshow(grayscale, cmap="gray")
    axes[1, column].imshow(cams[0], cmap="jet", alpha=0.45)
    axes[1, column].set_title(
        f"đoán: {CLASSES[row['pred']]} ({probability:.1%})", fontsize=11
    )
    for axis in axes[:, column]:
        axis.axis("off")

figure.suptitle(
    "Grad-CAM — vùng model dựa vào để quyết định.\n"
    "Nhiệt phải nằm trên nhu mô phổi, không phải mép ảnh hay ký hiệu R/L.",
    fontsize=13,
)
plt.tight_layout()
plt.show()

### 10.3. Kiểm tra định lượng — nhiệt có rơi ra rìa ảnh không?

Nhìn mắt thường vài tấm rất dễ tự huyễn hoặc. Đo thẳng trên một mẫu test: bao
nhiêu phần trăm khối lượng CAM rơi vào **viền ngoài 15%** của ảnh, nơi không thể
có nhu mô phổi.

So với **baseline** = tỉ lệ diện tích của chính viền đó (nếu nhiệt rải hoàn toàn
ngẫu nhiên thì hai số bằng nhau). Cao hơn baseline nghĩa là model đang bám
artifact, và mọi con số accuracy ở trên đều cần bị nghi ngờ.

In [ ]:
def border_mass_fraction(cam, border=BORDER_FRAC):
    """Fraction of CAM mass landing in the outer frame of the image."""
    height, width = cam.shape
    margin_y, margin_x = int(height * border), int(width * border)
    interior = cam[margin_y:height - margin_y, margin_x:width - margin_x].sum()
    return float(1.0 - interior / (cam.sum() + 1e-8))


sample = test_rows.sample(n=min(120, len(test_rows)), random_state=SEED)
fractions = []
for _, row in sample.iterrows():
    tensor = eval_tf(Image.open(row["path"]).convert("L")).unsqueeze(0).to(DEVICE)
    cams, _ = cam_engine(tensor)
    fractions.append(border_mass_fraction(cams[0]))

fractions = np.array(fractions)
baseline = 1 - (1 - 2 * BORDER_FRAC) ** 2

print(f"n = {len(fractions)} ảnh test | viền ngoài {BORDER_FRAC:.0%}")
print(f"  trung vị   : {np.median(fractions):.1%}")
print(f"  trung bình : {fractions.mean():.1%}")
print(f"  tệ nhất    : {fractions.max():.1%}")
print(f"  baseline   : {baseline:.1%}  (nhiệt rải đều thì bằng đúng số này)")
print(f"  số ca vượt baseline: {(fractions > baseline).sum()}/{len(fractions)}")
print()
if np.median(fractions) < baseline:
    print("=> Trung vị DƯỚI baseline: nhiệt tập trung vào giữa ảnh, đúng kỳ vọng.")
else:
    print("=> CẢNH BÁO: nhiệt dồn ra rìa nhiều hơn ngẫu nhiên — nghi model bám artifact.")

cam_engine.close()

## 11. Tóm tắt để đưa vào report

- **Số chính lấy từ protocol B** (`patient_grouped`). Số protocol A chỉ dùng để so
  với literature và **phải ghi rõ nhãn** khi trích dẫn.
- Artefact đã lưu ở `/kaggle/working/`: checkpoint `resnet18_<protocol>.pth` và
  manifest `manifest_<protocol>.csv`. Đính kèm manifest khi nộp để reviewer truy
  ngược được đúng danh sách file đứng sau mỗi con số.
- Grad-CAM nói **"model attends to"**, không phải "model detects lesion" —
  attention map không phải bằng chứng nhân quả.
- Tránh các từ "trustworthy", "clinical-ready", "radiologist-level" nếu chưa có
  external clinical validation.

### Việc còn lại
- Chạy nhiều seed (3–5) rồi báo cáo khoảng dao động, thay vì một con số đơn lẻ.
- So baseline này với DenseNet121 / EfficientNet-B0 trước khi kết luận kiến trúc nào hơn.
- Ca **FN** trong phần 10.2 là chỗ đáng soi nhất về mặt lâm sàng.